# Load EPA Air-Quality Data

This notebook is a Colab-friendly runner for an existing project module. The implementation remains in `src/`; this notebook provides an ordered, documented execution surface and writes outputs back into the repository.
Source implementation: [src/data/load_data.py](src/data/load_data.py)


In [1]:
# Local project setup
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")




Project root: g:\ACHINI\sliit activities\Y3S2\Data Science\FDM\FDM-Mini-Project


## Run the stage

Memory-conscious loader for the EPA annual air-quality CSV.

In [2]:

from __future__ import annotations
from pathlib import Path
from typing import Iterator
import pandas as pd

DEFAULT_RAW_PATH = project_root / "data" / "raw" / "air_quality.csv"

# Keep identifiers as strings so leading zeroes are not lost.
ID_DTYPES = {
    "state_code": "string",
    "county_code": "string",
    "site_num": "string",
    "parameter_code": "Int64",
    "poc": "Int64",
    "year": "Int64",
}

def load_air_quality(
    path: str | Path = DEFAULT_RAW_PATH,
    *,
    chunksize: int | None = None,
    usecols: list[str] | None = None,
) -> pd.DataFrame | Iterator[pd.DataFrame]:
    """Load the EPA CSV. If chunksize is supplied, return a TextFileReader."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"EPA dataset not found: {path}. Put the full CSV at data/raw/air_quality.csv "
            "or pass a different path."
        )

    kwargs = dict(
        low_memory=False,
        dtype=ID_DTYPES,
        usecols=usecols,
    )
    if chunksize:
        return pd.read_csv(path, chunksize=chunksize, **kwargs)
    return pd.read_csv(path, **kwargs)

def iter_air_quality(
    path: str | Path = DEFAULT_RAW_PATH,
    *,
    chunksize: int = 250_000,
    usecols: list[str] | None = None,
) -> Iterator[pd.DataFrame]:
    """Yield chunks; preferred for the full ~300 MB file."""
    yield from load_air_quality(path, chunksize=chunksize, usecols=usecols)

if __name__ == "__main__":
    df = load_air_quality()
    print(f"Loaded shape: {df.shape}")
    print(df.head())



Loaded shape: (2038710, 55)
  state_code county_code site_num  parameter_code  poc   latitude   longitude  \
0         20         057     0001           82134    1  37.771964 -100.018190   
1         18         163     0012           42401    1  38.021727  -87.569458   
2         40         031     9000           88152    1  34.732404  -98.713351   
3         32         003     0561           42601    1  36.163959 -115.113916   
4         13         127     3001           12154    1  31.185187  -81.485166   

   datum       parameter_name sample_duration  ... seventy_five_percentile  \
0  WGS84  Molybdenum PM10 STP         24 HOUR  ...                 0.00000   
1  WGS84       Sulfur dioxide    3-HR BLK AVG  ...                 3.60000   
2  WGS84  Phosphorus PM2.5 LC         24 HOUR  ...                 0.00000   
3  WGS84    Nitric oxide (NO)          1 HOUR  ...                18.80000   
4  WGS84   Selenium (TSP) STP         24 HOUR  ...                 0.00064   

  fifty_percenti